<a href="https://colab.research.google.com/github/AmatHub21/codingan_kelompok_mechineLearning/blob/main/Codingan%20pertemuan%204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import mutual_info_classif
from scipy import stats

# Memuat dataset Heart Disease Cleveland
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope",
    "ca", "thal", "target"
]

df = pd.read_csv(url, header=None, names=columns, na_values="?")

# Binerisasi target (0 = sehat, >0 = terindikasi penyakit jantung)
df["target"] = (df["target"] > 0).astype(int)

In [3]:
X = df.drop(columns="target")
y = df["target"]

# ============================================================
# SEBELUM MITIGASI (DATA LEAKAGE): fit_transform sebelum train_test_split
# ============================================================

scaler_bocor = StandardScaler()

# >>> KODE INTI PEMBUKTIAN MASALAH (FITUR KESELURUHAN DITRANSFORMASI TERLEBIH DAHULU) <<<
X_bocor = scaler_bocor.fit_transform(X.fillna(X.median()))
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

X_tr_l, X_ts_l, y_tr_l, y_ts_l = train_test_split(
    X_bocor, y, test_size=0.2, random_state=42
)

model_bocor = LogisticRegression()
model_bocor.fit(X_tr_l, y_tr_l)
acc_sebelum = model_bocor.score(X_ts_l, y_ts_l)

# ============================================================
# SESUDAH MITIGASI (ISOLASI DENGAN PIPELINE): fit hanya pada data latih
# ============================================================

X_tr, X_ts, y_tr, y_ts = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
# >>> KODE INTI PEMECAHAN MASALAH (ISOLASI LEWAT PIPELINE SCIKIT-LEARN) <<<

pipeline_benar = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

pipeline_benar.fit(X_tr.fillna(X_tr.median()), y_tr)

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

acc_sesudah = pipeline_benar.score(
    X_ts.fillna(X_tr.median()), y_ts
)

print(f"[Sebelum / Leaked] Akurasi Data Uji: {acc_sebelum:.4f}")
print(f"[Sesudah / Isolasi] Akurasi Data Uji: {acc_sesudah:.4f}")

[Sebelum / Leaked] Akurasi Data Uji: 0.8852
[Sesudah / Isolasi] Akurasi Data Uji: 0.8852


libary yang terdapat di code :

Certainly! Based on the provided code, the following libraries are used:

numpy (as np),
pandas (as pd),
matplotlib.pyplot (as plt),
seaborn (as sns),
sklearn.model_selection (specifically, train_test_split),
sklearn.preprocessing (specifically StandardScaler, RobustScaler),
sklearn.linear_model (specifically LogisticRegression),
sklearn.pipeline (specifically Pipeline),
sklearn.metrics (specifically accuracy_score, f1_score, classification_report),
statsmodels.stats.outliers_influence (specifically variance_inflation_factor),
sklearn.feature_selection (specifically mutual_info_classif),
scipy (specifically stats).

You're right to ask again, it's a very good observation! The fact that the test accuracies are identical (0.8852) for both the 'leaked' and 'isolated' scenarios in this particular case is not necessarily what you'd always expect, but it can happen for a few reasons:

Dataset Characteristics: The distribution of features in this specific Heart Disease Cleveland dataset might be quite uniform, meaning that the scaling parameters (mean and standard deviation for StandardScaler or median for fillna) derived from the full dataset are very similar to those derived solely from the training set. If these parameters are almost identical, the transformations applied to the test data will yield virtually the same scaled values, leading to the same model performance.
Model Robustness: Logistic Regression is a relatively robust model. While sensitive to feature scaling, it might not be drastically impacted by subtle differences in scaling parameters for this dataset. More complex models or models more sensitive to input scale might show a greater difference.
Limited Data: This dataset isn't extremely large. With more data, especially if there are significant outliers or distributional shifts between the training and test sets, the difference caused by data leakage might become more apparent.
Important Takeaway: Even though the numerical result is the same here, the 'isolated' approach (using a Pipeline to ensure scaling and imputation are fitted only on the training data) is the correct and generally recommended methodology. It prevents potential bias and ensures your model's performance on unseen data is truly representative. In other datasets or with different models, data leakage could lead to an overly optimistic (and therefore misleading) accuracy score. This example simply demonstrates a scenario where the practical impact on this specific metric is minimal.

In [6]:
# Cek kemencengan awal
skew_chol = df["chol"].skew()
print(f"Koefisien Kemencengan Awal (chol): {skew_chol:.3f}")

# ============================================================
# SEBELUM MITIGASI: StandardScaler tanpa normalisasi bentuk sebaran
# ============================================================

scaler_std = StandardScaler()

# >>> KODE INTI BUKTI KEGAGALAN STANDARDSCALER <<<
chol_std = scaler_std.fit_transform(df[["chol"]])
skew_sebelum = pd.Series(chol_std.flatten()).skew()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>


# ============================================================
# SESUDAH MITIGASI: Transformasi Logaritmik dan RobustScaler
# ============================================================

# >>> KODE INTI PEMECAHAN MASALAH (LOG TRANSFORMATION & ROBUST SCALING) <<<
chol_log = np.log1p(df["chol"])
chol_robust = RobustScaler().fit_transform(chol_log.to_frame())
skew_sesudah = pd.Series(chol_log).skew()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>


print(f"[Sebelum] Nilai Skewness setelah StandardScaler : {skew_sebelum:.3f}")
print(f"[Sesudah] Nilai Skewness setelah Log Transform : {skew_sesudah:.3f}")

Koefisien Kemencengan Awal (chol): 1.136
[Sebelum] Nilai Skewness setelah StandardScaler : 1.136
[Sesudah] Nilai Skewness setelah Log Transform : 0.082
